# Assignment 3 — Two Averages, One Argument
**The Prices Desk | Week 3 — Descriptive analytics**

This notebook uses the actual `teaching_prices_messy.csv` uploaded for this assignment and reapplies the cleaning decisions documented in Assignment 2. It preserves the raw price, removes one exact duplicate only from the analytical table, parses comma-formatted numbers, and retains uncertain observations with flags.

**Data label:** illustrative teaching data — not actual PBS observations.


In [2]:
import pandas as pd
import numpy as np

raw = pd.read_csv("teaching_prices_messy.csv", dtype={"price": "string"})
df = raw.copy()
df["price_raw"] = df["price"]
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["item_clean"] = (df["item"].astype("string").str.strip()
                    .str.replace(r"\\s+", " ", regex=True).str.title())
df["price_num"] = pd.to_numeric(
    df["price_raw"].str.replace(",", "", regex=False), errors="coerce"
)
df["exact_duplicate"] = df.duplicated(keep="first")
df["missing_price"] = df["price_num"].isna()
df["extreme_9999"] = df["price_num"].eq(9999)
df["suspicious_oil_city1_low"] = (
    df["item_clean"].eq("Oil B") & df["city"].eq("City 1") & df["price_num"].lt(200)
)
df["comma_formatted"] = df["price_raw"].str.contains(",", na=False)
df["review_flag"] = df[["missing_price", "extreme_9999",
                        "suspicious_oil_city1_low"]].any(axis=1)

audit = df.copy()
clean = df.loc[~df["exact_duplicate"]].copy()
print("Raw rows:", len(raw), "| Analytical rows:", len(clean))
print("Missing prices:", clean["price_num"].isna().sum())
print("Flagged 9999:", clean["extreme_9999"].sum())
print("Suspicious low Oil B / City 1:", clean["suspicious_oil_city1_low"].sum())
print("Comma-formatted parsed values:", clean.loc[clean["comma_formatted"], "price_num"].tolist())


Raw rows: 97 | Analytical rows: 96
Missing prices: 1
Flagged 9999: 1
Suspicious low Oil B / City 1: 1
Comma-formatted parsed values: [1250.0]


## 1. Mean, median and first-to-last percentage change

Mean and median use all available numeric prices for each item in the cleaned analytical table. Missing prices are omitted; flagged numeric values remain included.

For the first-to-last change, calculate the mean of the available city prices for that item on its first date and last date, then:

\[
\\text{Change (\\%)}=\\frac{\\text{last-date mean}-\\text{first-date mean}}{\\text{first-date mean}}\\times100
\]


In [3]:
summary_rows = []
for item, group in clean.groupby("item_clean", sort=True):
    values = group["price_num"].dropna()
    date_means = group.groupby("date")["price_num"].mean().sort_index()
    first_mean, last_mean = date_means.iloc[0], date_means.iloc[-1]
    summary_rows.append({
        "Item": item, "Numeric observations": len(values),
        "Mean (PKR/kg)": values.mean(), "Median (PKR/kg)": values.median(),
        "First-date mean": first_mean, "Last-date mean": last_mean,
        "First-to-last change (%)": (last_mean-first_mean)/first_mean*100
    })
summary = pd.DataFrame(summary_rows)
display(summary.round(2))
summary.to_csv("item_summary.csv", index=False)


,Item,Numeric observations,Mean (PKR/kg),Median (PKR/kg),First-date mean,Last-date mean,First-to-last change (%)
0,Oil B,25,316.18,324.00,301.0,347.0,15.28
1,Pulse C,23,205.91,206.00,200.0,210.5,5.25
2,Staple A,23,160.89,112.50,104.0,117.5,12.98
3,Sweetener D,24,576.81,167.75,152.0,181.5,19.41


## 2. One flagged value: Sweetener D with and without 9999

The 9999 value remains in the main cleaned data because its intended correction is unknown. Excluding it below is only a sensitivity comparison, not a correction.


In [4]:
sweetener = clean.loc[clean["item_clean"].eq("Sweetener D"), "price_num"].dropna()
without_9999 = sweetener.loc[sweetener.ne(9999)]
sensitivity = pd.DataFrame([
    {"Calculation":"Including flagged 9999","n":len(sweetener),
     "Mean":sweetener.mean(),"Median":sweetener.median()},
    {"Calculation":"Excluding flagged 9999 for sensitivity check","n":len(without_9999),
     "Mean":without_9999.mean(),"Median":without_9999.median()}
])
display(sensitivity.round(2))
sensitivity.to_csv("sweetener_sensitivity.csv", index=False)


,Calculation,n,Mean,Median
0,Including flagged 9999,24,576.81,167.75
1,Excluding flagged 9999 for sensitivity check,23,167.15,167.50


## 3. Review flagged rows and export files


In [5]:
display(clean.loc[
    clean["review_flag"] | clean["comma_formatted"],
    ["date","item","item_clean","city","price_raw","price_num",
     "missing_price","extreme_9999","suspicious_oil_city1_low","comma_formatted"]
].sort_values(["date","item_clean","city"]))
clean.to_csv("teaching_prices_cleaned_with_flags.csv", index=False)
audit.to_csv("teaching_prices_audit_all_rows.csv", index=False)
print("Exports saved.")


,date,item,item_clean,city,price_raw,price_num,missing_price,extreme_9999,suspicious_oil_city1_low,comma_formatted
5,2026-01-01,Pulse C,Pulse C,City 2,<NA>,<NA>,True,<NA>,False,False
17,2026-01-15,Staple A,Staple A,City 2,"1,250",1250.0,False,False,False,True
30,2026-01-22,Sweetener D,Sweetener D,City 1,9999,9999.0,False,True,False,False
40,2026-02-05,oil b,Oil B,City 1,108.5,108.5,False,False,True,False


Exports saved.
